# Quick Start: Running a Feed Forward Model on GIFT-Eval


This notebook demonstrates how to run a GluonTS [Simple Feed Forward model](https://ts.gluon.ai/dev/api/gluonts/gluonts.torch.model.simple_feedforward.html) on the GIFT-Eval benchmark.


## Dataset


First, let's load the dataset. For the sake of brevity, we'll only load two datasets.


In [6]:
import json
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create a set of all the short dataset names
short_dataset_names = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
short_datasets = set(short_dataset_names.split())

# Name of the short dataset we'll use
short_dataset = "m4_hourly"

# Create a set of all the medium to long dataset names
med_long_dataset_names = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets = set(med_long_dataset_names.split())

# Name of the medium to long dataset we'll use
med_long_dataset = "bizitobs_l2c/H"

# Combine the datasets into one list
all_datasets = [short_dataset, med_long_dataset]

# Load the dataset properties map
dataset_properties_map = json.load(open("dataset_properties.json"))

## Training


Initialize a CSV file to save our model's performance on each dataset


In [1]:
import csv
import os

# Name of the directory where our model's results will be saved
output_directory = "./results/feedforward"

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the CSV file's path
csv_file_path = os.path.join(output_directory, "results.csv")

# Initialize the CSV file's header
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Write the header
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
        ]
    )

Load the metrics we'll evaluate our model on


In [8]:
from gluonts.ev.metrics import (
    MSE,
    MAE,
    MASE,
    MAPE,
    SMAPE,
    MSIS,
    RMSE,
    NRMSE,
    ND,
    MeanWeightedSumQuantileLoss,
)

# Instantiate metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

Before training and evaluation, let's define some helper functions


In [9]:
from gluonts.model import evaluate_model
from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator
from gluonts.time_feature import get_seasonality
from notebooks.src.gift_eval.data import Dataset

terms = ["short", "medium", "long"]


def get_dataset_key(dataset_name):
    return dataset_name.split("/")[0] if "/" in dataset_name else dataset_name.lower()


def get_dataset_freq(dataset_name):
    key = get_dataset_key(dataset_name)
    return (
        dataset_name.split("/")[1]
        if "/" in dataset_name
        else dataset_properties_map[key]["frequency"]
    )


def get_dataset_config(dataset_name, term):
    key = get_dataset_key(dataset_name)
    freq = get_dataset_freq(dataset_name)
    return f"{key}/{freq}/{term}"


def write_to_csv(csv_file_path, dataset_name, term, results):
    # Initialize dataset's configuation
    dataset_config = get_dataset_config(dataset_name, term)

    # Write the results to the CSV file
    with open(csv_file_path, "a", newline="") as csvfile:
        key = get_dataset_key(dataset_name)

        writer = csv.writer(csvfile)
        writer.writerow(
            [
                dataset_config,
                "feedforward",
                results["MSE[mean]"][0],
                results["MSE[0.5]"][0],
                results["MAE[0.5]"][0],
                results["MASE[0.5]"][0],
                results["MAPE[0.5]"][0],
                results["sMAPE[0.5]"][0],
                results["MSIS"][0],
                results["RMSE[mean]"][0],
                results["NRMSE[mean]"][0],
                results["ND[0.5]"][0],
                results["mean_weighted_sum_quantile_loss"][0],
                dataset_properties_map[key]["domain"],
                dataset_properties_map[key]["num_variates"],
            ]
        )


def check_if_multivariate(dataset_name, term):
    # Get the dataset's target dimension
    target_dimension = Dataset(
        name=dataset_name,
        term=term,
        to_univariate=False,
    ).target_dim

    # Check if the dataset is already univariate
    return target_dimension > 1


def train_eval(dataset_name, term, csv_file_path):
    # Check if the dataset is multivariate
    is_multivariate = check_if_multivariate(dataset_name, term)

    # True if the dataset is multivariate
    to_univariate = True if is_multivariate else False

    # Initialize dataset
    dataset = Dataset(name=dataset_name, term=term, to_univariate=to_univariate)

    # Get the seasonal component's length
    season_length = get_seasonality(dataset.freq)

    # Get the dataset's prediction length
    prediction_length = dataset.prediction_length

    # Define hyperparameters
    trainer_kwargs = {"max_epochs": 1}

    # Instantiate the model
    estimator = SimpleFeedForwardEstimator(
        prediction_length=prediction_length,
        context_length=prediction_length,
        trainer_kwargs=trainer_kwargs,
    )

    # Train the model on the validation set
    predictor = estimator.train(dataset.validation_dataset)

    # Evaluate the model on the test set
    results = evaluate_model(
        predictor,
        test_data=dataset.test_data,
        metrics=metrics,
        batch_size=512,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=season_length,
    )

    write_to_csv(csv_file_path, dataset_name, term, results)

    print(f"Results for {dataset_name} have been written to {csv_file_path}")

Now, we'll train and evaluate the model on each dataset


In [10]:
# Train and evaluate the model on all of the datasets
for dataset_name in all_datasets:
    print(f"Processing dataset: {dataset_name}")

    for term in terms:
        if term != "short" and dataset_name not in med_long_datasets:
            continue
        train_eval(dataset_name, term, csv_file_path)

c:\Users\Mike\anaconda3\envs\gift\Lib\site-packages\gluonts\time_feature\seasonality.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset = pd.tseries.frequencies.to_offset(freq)
c:\Users\Mike\vscode\gift-eval\notebooks\src\gift_eval\data.py:134: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  freq = norm_freq_str(to_offset(self.freq).name)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\Mike\anaconda3\envs\gift\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages ar

Processing dataset: m4_hourly
M4_PRED_LENGTH_MAP keys: dict_keys(['A', 'Q', 'M', 'W', 'D', 'H'])
Epoch 0: |          | 50/? [00:00<00:00, 83.67it/s, v_num=19, train_loss=6.120]

Epoch 0, global step 50: 'train_loss' reached 6.12437 (best 6.12437), saving model to 'c:\\Users\\Mike\\vscode\\gift-eval\\lightning_logs\\version_19\\checkpoints\\epoch=0-step=50.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [00:00<00:00, 82.77it/s, v_num=19, train_loss=6.120]


414it [00:07, 55.43it/s]
C:\Users\Mike\AppData\Local\Temp\ipykernel_29820\3649893345.py:41: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results["MSE[mean]"][0],
C:\Users\Mike\AppData\Local\Temp\ipykernel_29820\3649893345.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results["MSE[0.5]"][0],
C:\Users\Mike\AppData\Local\Temp\ipykernel_29820\3649893345.py:43: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results["MAE[0.5]"][0],

...

Results for bizitobs_l2c/H have been written to ./results/feedforward\results.csv
PRED_LENGTH_MAP keys: dict_keys(['M', 'W', 'D', 'H', 'T', 'S'])
Epoch 0: |          | 50/? [00:00<00:00, 81.14it/s, v_num=22, train_loss=4.830]

Epoch 0, global step 50: 'train_loss' reached 4.83235 (best 4.83235), saving model to 'c:\\Users\\Mike\\vscode\\gift-eval\\lightning_logs\\version_22\\checkpoints\\epoch=0-step=50.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [00:00<00:00, 79.64it/s, v_num=22, train_loss=4.830]


7it [00:00, 18.74it/s]

Results for bizitobs_l2c/H have been written to ./results/feedforward\results.csv


## Results

Running the above cell will generate a csv file called `all_results.csv` under the `results/feedforward` folder containing the results for the Feed Forward model on the gift-eval benchmark. The csv file will look like this:


In [11]:
import pandas as pd
from datetime import datetime

# Get the current timestamp
timestamp = datetime.now()
print(f"Printing results: {timestamp}")

df = pd.read_csv("./results/feedforward/all_results.csv")
df.head()

Printing results: 2025-03-01 16:41:12.022155


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
